# ALNS + RL Repair cho bài toán Thiết kế Lãnh thổ (DTDP)

---

## Tổng quan luồng xử lý

```
┌──────────────────────────────────────────────────────────────────────┐
│                         TOÀN BỘ PIPELINE                            │
│                                                                      │
│  Bước 0 ── Cài đặt thư viện, đường dẫn, tham số                    │
│  Bước 1 ── Khám phá dữ liệu (thống kê + visualize graph)            │
│  Bước 2 ── Định nghĩa hàm tiện ích                                  │
│                                                                      │
│  ┌──── TGraph Pipeline ─────────────────────────────────────────┐   │
│  │  Train  : planar500 G0-G9         → model_tgraph.npz         │   │
│  │  Eval   : planar500 + planar600 + planar700  (mỗi 10 inst.)  │   │
│  └──────────────────────────────────────────────────────────────┘   │
│                                                                      │
│  ┌──── GGraph Pipeline ─────────────────────────────────────────┐   │
│  │  Train  : Center486 G0-G9         → model_ggraph.npz         │   │
│  │  Eval   : 27x27  (486) × Center / Corners / Diagonal         │   │
│  │           30x30  (600) × Center / Corners / Diagonal         │   │
│  │           33x33  (726) × Center / Corners / Diagonal         │   │
│  │           = 9 nhóm × 10 instances = 90 instances             │   │
│  └──────────────────────────────────────────────────────────────┘   │
│                                                                      │
│  Bước 7 ── So sánh ALNS+RL vs VNS  (bảng + biểu đồ)               │
└──────────────────────────────────────────────────────────────────────┘
```

---

## Thiết kế thực nghiệm

### TGraph — Đồ thị phẳng ngẫu nhiên
| Tập | Instances | Mục đích |
|---|---|---|
| Train | planar**500** G0-G9 | Học chiến lược repair trên graph nhỏ |
| Eval  | planar**500** G0-G9 | Đánh giá in-distribution |
| Eval  | planar**600** G0-G9 | Generalization: graph lớn hơn (+20%) |
| Eval  | planar**700** G0-G9 | Generalization: graph lớn hơn (+40%) |

### GGraph — Đồ thị lưới
| Tập | Instances | Mục đích |
|---|---|---|
| Train | **Center486** G0-G9 | Học chiến lược trên lưới 27×27, depot trung tâm |
| Eval  | Center/Corners/Diagonal **486** | In-distribution (cùng kích thước) |
| Eval  | Center/Corners/Diagonal **600** | Generalization: lưới lớn hơn (30×30) |
| Eval  | Center/Corners/Diagonal **726** | Generalization: lưới lớn hơn (33×33) |

→ GGraph đánh giá generalization trên **2 chiều**: loại depot (Center/Corners/Diagonal) × kích thước lưới (486/600/726)

---

## Tham số thực nghiệm

| Tham số | Giá trị | Ghi chú |
|---|---|---|
| `llambda` | 0.7 | Trọng số diameter vs infeasibility trong merit |
| `delta` | 0.05 | Độ lệch cho phép giữa các district (±5%) |
| `nr_districts` | 10 | Số lãnh thổ cần phân chia |
| `N_ITER` | 300 | Số vòng lặp ALNS mỗi instance |
| `K_REMOVE` | 10 | Số node xóa mỗi bước destroy |

---
## Bước 0 — Cài đặt

Import thư viện, khai báo tất cả đường dẫn, đặt tham số.

**Output mong đợi:** `Setup OK`

In [ ]:
import os, json, time, warnings, copy
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from dtdp import TerritoryDesignProblem
from alns_rl_repair import alns_rl_repair, RepairAgent, activities

warnings.filterwarnings('ignore')

# ── Dữ liệu đầu vào ──────────────────────────────────────────────
TGRAPH_INST  = 'TGraphInstances'
GGRAPH_BASE  = 'GGraphInstances/newGeneratedInstances'

# ── Kết quả VNS baseline ─────────────────────────────────────────
VNS_PLANAR   = 'Results/VNSExperiments'
VNS_GGRAPH   = 'Results/VNSExperimentsGeneral'

# ── Lưu kết quả ALNS+RL ──────────────────────────────────────────
OUT_PLANAR   = 'Results/MyExperiments'
OUT_GGRAPH   = 'Results/MyExperimentsGeneral'

# ── Model ALNS+RL (tách riêng TGraph / GGraph) ───────────────────
MODEL_TGRAPH = 'model_tgraph.npz'
MODEL_GGRAPH = 'model_ggraph.npz'

# ── Tất cả kích thước GGraph ─────────────────────────────────────
GSIZES = [
    {'folder': '27x27Graphs', 'suffix': '486'},
    {'folder': '30x30Graphs', 'suffix': '600'},
    {'folder': '33x33Graphs', 'suffix': '726'},
]
GTYPES = ['Center', 'Corners', 'Diagonal']

# Tạo thư mục đầu ra
os.makedirs(OUT_PLANAR, exist_ok=True)
for gs in GSIZES:
    os.makedirs(f"{OUT_GGRAPH}/{gs['folder']}", exist_ok=True)

# ── Tham số ──────────────────────────────────────────────────────
LMBDA        = 0.7
DELTA        = 0.05
NR_DISTRICTS = 10
N_ITER       = 300   # số vòng lặp ALNS mỗi instance
K_REMOVE     = 10    # số node xóa mỗi bước destroy

print('Setup OK')

---
## Bước 1 — Khám phá dữ liệu

### 1a. Thống kê tất cả instances

Liệt kê toàn bộ file `.graphml` có sẵn kèm số node, cạnh để xác nhận dữ liệu đầy đủ trước khi chạy.

**Tổng:** 30 TGraph + 90 GGraph = **120 instances**

In [ ]:
print(f"{'Instance':<26} {'Type':>8} {'Grid':>8} {'Nodes':>6} {'Edges':>7}")
print('-' * 60)

total = 0
# TGraph
for size in [500, 600, 700]:
    cnt = sum(1 for i in range(10)
              if os.path.exists(f'{TGRAPH_INST}/planar{size}_G{i}.graphml'))
    if cnt:
        fp = f'{TGRAPH_INST}/planar{size}_G0.graphml'
        G  = nx.read_graphml(fp)
        print(f"planar{size}_G0..G{cnt-1:<10} {'TGraph':>8} {'planar':>8} "
              f"{G.number_of_nodes():>6} {G.number_of_edges():>7}  ({cnt} inst.)")
        total += cnt

print()
# GGraph
for gs in GSIZES:
    folder = gs['folder']
    suffix = gs['suffix']
    base   = f"{GGRAPH_BASE}/{folder}"
    for tp in GTYPES:
        cnt = sum(1 for i in range(10)
                  if os.path.exists(f"{base}/{tp}{suffix}_G{i}.graphml"))
        if cnt:
            fp = f"{base}/{tp}{suffix}_G0.graphml"
            G  = nx.read_graphml(fp)
            print(f"{tp}{suffix}_G0..G{cnt-1:<10} {'GGraph':>8} {folder:>8} "
                  f"{G.number_of_nodes():>6} {G.number_of_edges():>7}  ({cnt} inst.)")
            total += cnt
    print()

print(f'Tổng: {total} instances')

### 1b. Visualize cấu trúc graph

So sánh trực quan cấu trúc đồ thị của TGraph và các loại GGraph.

- **TGraph (planar):** phân bố ngẫu nhiên, không có hướng cụ thể
- **GGraph lưới:** node xếp đều trên lưới vuông, khác nhau ở vị trí depot

In [ ]:
def get_pos(G):
    """Layout: dùng x/y nếu có (TGraph), suy ra lưới từ ID node (GGraph)."""
    sample_attrs = next(iter(G.nodes(data=True)))[1]
    if 'x' in sample_attrs and 'y' in sample_attrs:
        return {n: (float(d['x']), float(d['y'])) for n, d in G.nodes(data=True)}
    # GGraph: tìm width lưới từ neighbor nhỏ nhất của node 0 (> 1)
    try:
        nodes_int = [int(n) for n in G.nodes()]
        node0_id  = [n for n in G.nodes() if int(n) == min(nodes_int)][0]
        nbrs      = sorted([int(nb) for nb in G.neighbors(node0_id) if int(nb) > 1])
        width     = nbrs[0]  # e.g., 27 for 27x27 grid
        return {n: (int(n) % width, int(n) // width) for n in G.nodes()}
    except Exception:
        return nx.spring_layout(G, seed=42)

samples = [
    (f'{TGRAPH_INST}/planar500_G0.graphml',
     'TGraph\nplanar500'),
    (f'{GGRAPH_BASE}/27x27Graphs/Center486_G0.graphml',
     'GGraph 27x27\nCenter486'),
    (f'{GGRAPH_BASE}/27x27Graphs/Corners486_G0.graphml',
     'GGraph 27x27\nCorners486'),
    (f'{GGRAPH_BASE}/27x27Graphs/Diagonal486_G0.graphml',
     'GGraph 27x27\nDiagonal486'),
    (f'{GGRAPH_BASE}/30x30Graphs/Center600_G0.graphml',
     'GGraph 30x30\nCenter600'),
    (f'{GGRAPH_BASE}/33x33Graphs/Center726_G0.graphml',
     'GGraph 33x33\nCenter726'),
]

fig, axes = plt.subplots(1, len(samples), figsize=(20, 4))
for ax, (fp, title) in zip(axes, samples):
    G   = nx.read_graphml(fp)
    pos = get_pos(G)
    nx.draw(G, pos, ax=ax, node_size=5, node_color='steelblue',
            edge_color='#bbb', width=0.3, with_labels=False)
    ax.set_title(f'{title}\n({G.number_of_nodes()}n)', fontsize=8)

plt.suptitle('Cấu trúc graph: TGraph (ngẫu nhiên) vs GGraph (lưới)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Bước 2 — Hàm tiện ích

| Hàm | Mô tả |
|---|---|
| `make_tdp(G)` | Tạo `TerritoryDesignProblem` từ graph G với params chuẩn |
| `make_bvns(tdp)` | Tạo `BVNS` với params khớp VNS |
| `train_on_instance(fp, model)` | Train RL agent 1 instance, lưu model, **không** lưu kết quả eval |
| `eval_instance(fp, name, dir, model)` | Chạy BVNS+RL không train, lưu JSON, trả về dict kết quả |
| `load_json(path)` | Đọc file JSON nếu tồn tại, trả về `None` nếu không có |

> **Tại sao tách train và eval thành 2 hàm riêng?**  
> Trong training: `train_agent=True` → RL cập nhật trọng số DQN sau mỗi iteration.  
> Trong eval: `train_agent=False` → RL chỉ ra quyết định dựa trên model đã học, trọng số không thay đổi → kết quả tái lập được.

In [ ]:
def make_tdp(G):
    return TerritoryDesignProblem(
        graph_input=G, delta=DELTA, llambda=LMBDA,
        rcl_parameter=0.2, nr_districts=NR_DISTRICTS
    )

def make_agent(tdp):
    return RepairAgent(
        n_features=len(activities)*2 + 2,
        n_actions=tdp.nr_districts,
        alpha=0.01, gamma=0.9,
        epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995
    )

def train_on_instance(file_path, model_path):
    """Train ALNS+RL agent trên 1 instance; tích lũy vào model hiện tại."""
    G     = nx.read_graphml(file_path)
    tdp   = make_tdp(G)
    agent = make_agent(tdp)
    agent.load(model_path)
    alns_rl_repair(tdp, agent=agent, n_iterations=N_ITER,
                   k_remove=K_REMOVE, lam=LMBDA, seed=0, train=True)
    agent.save(model_path)

def eval_instance(file_path, instance_name, save_dir, model_path):
    """Eval ALNS+RL (không train); lưu JSON; trả về dict kết quả."""
    t0    = time.time()
    G     = nx.read_graphml(file_path)
    tdp   = make_tdp(G)
    agent = make_agent(tdp)
    agent.load(model_path)
    agent.epsilon = agent.epsilon_min   # greedy hoàn toàn khi eval
    best_F, best_G, best_sol, history, _ = alns_rl_repair(
        tdp, agent=agent, n_iterations=N_ITER,
        k_remove=K_REMOVE, lam=LMBDA, seed=42, train=False)
    elapsed = time.time() - t0

    result = {
        'Instance':      instance_name,
        'Objective':     best_F,
        'Infeasibility': best_G,
        'Time Taken':    elapsed,
        'nrDistricts':   len(best_sol),
        'delta':         DELTA,
        'llambda':       LMBDA,
        'Allocation':    {str(k): [str(n) for n in v]
                          for k, v in best_sol.items()},
        '_obj_hist':     history['obj'],
        '_inf_hist':     history['inf'],
    }
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f'{instance_name}.json'), 'w') as f:
        json.dump({k: v for k, v in result.items() if not k.startswith('_')},
                  f, indent=4)
    return result

def load_json(path):
    return json.load(open(path)) if os.path.exists(path) else None

print('Helpers defined.')

---
# TGRAPH PIPELINE

```
planar500 G0-G9 ──train──► model_tgraph.pt
                                  │
                 ┌────────────────┼────────────────┐
                 ▼                ▼                ▼
          planar500 G0-G9  planar600 G0-G9  planar700 G0-G9
        (in-distribution) (generalize +20%) (generalize +40%)
```

## Bước 3 — TGraph: Training (planar500 G0–G9)

**Quy trình mỗi instance:**  
Đọc graph → Tạo TDP + BVNS → Load model → `performBVNS(train_agent=True)` → Lưu model cập nhật.

RL agent học cách chọn `k` (shaking step) và `λ` sao cho BVNS cải thiện solution nhanh nhất.

**Thời gian ước tính:** ~30–50 phút

In [ ]:
print('Training ALNS+RL on planar500 G0-G9 ...')
if os.path.exists(MODEL_TGRAPH):
    print(f'  (Tiếp tục từ model hiện có: {MODEL_TGRAPH})')
else:
    print(f'  (Khởi tạo model mới)')

print(f"{'#':<4} {'Instance':<18} {'Time':>7}")
print('-' * 32)
for i in range(10):
    fp = f'{TGRAPH_INST}/planar500_G{i}.graphml'
    t0 = time.time()
    print(f"{i+1:<4} planar500_G{i:<7} ...", end=' ', flush=True)
    train_on_instance(fp, MODEL_TGRAPH)
    print(f'{time.time()-t0:.0f}s')

print(f'\nTGraph model saved → {MODEL_TGRAPH}')

## Bước 4 — TGraph: Đánh giá (planar500 + 600 + 700)

Load `model_tgraph.pt` đã train, chạy `performBVNS(train_agent=False)` trên 30 instances.  
Kết quả lưu tại `Results/MyExperiments/{name}.json`.

**Thời gian ước tính:** ~90–150 phút (30 instances)

In [ ]:
tgraph_results = {}
print(f"{'#':<4} {'Instance':<20} {'Obj':>8} {'Inf':>8} {'Time':>7}")
print('-' * 52)
idx = 0

for size in [500, 600, 700]:
    for i in range(10):
        name = f'planar{size}_G{i}'
        fp   = f'{TGRAPH_INST}/{name}.graphml'
        if not os.path.exists(fp):
            continue
        idx += 1
        r = eval_instance(fp, name, OUT_PLANAR, MODEL_TGRAPH)
        tgraph_results[name] = r
        print(f"{idx:<4} {name:<20} {r['Objective']:>8.3f} "
              f"{r['Infeasibility']:>8.4f} {r['Time Taken']:>6.0f}s")

print(f'\nTGraph eval done: {len(tgraph_results)} instances.')

---
# GGRAPH PIPELINE

```
Center486 G0-G9 ──train──► model_ggraph.pt
                                  │
          ┌───────────────────────┼───────────────────────┐
          ▼                       ▼                       ▼
    27x27 (486 nodes)       30x30 (600 nodes)       33x33 (726 nodes)
  Center / Corners / Diag  Center / Corners / Diag  Center / Corners / Diag
  (in-dist + cross-type)   (cross-size + cross-type) (cross-size + cross-type)
```

Eval **2 chiều generalization:**
- **Cross-type:** Center → Corners, Diagonal (cùng kích thước, khác vị trí depot)
- **Cross-size:** 486 → 600 → 726 (cùng loại depot, graph lớn hơn)

## Bước 5 — GGraph: Training (Center486 G0–G9)

Khởi tạo model mới (`model_ggraph.pt`) riêng biệt với TGraph.  
Train trên Center486 — cấu hình đơn giản nhất (depot ở trung tâm lưới 27×27).

**Thời gian ước tính:** ~20–40 phút

In [ ]:
print('Training ALNS+RL on Center486 G0-G9 ...')
if os.path.exists(MODEL_GGRAPH):
    print(f'  (Tiếp tục từ model hiện có: {MODEL_GGRAPH})')
else:
    print(f'  (Khởi tạo model mới)')

print(f"{'#':<4} {'Instance':<20} {'Time':>7}")
print('-' * 34)
base = f"{GGRAPH_BASE}/27x27Graphs"
for i in range(10):
    fp = f'{base}/Center486_G{i}.graphml'
    t0 = time.time()
    print(f"{i+1:<4} Center486_G{i:<9} ...", end=' ', flush=True)
    train_on_instance(fp, MODEL_GGRAPH)
    print(f'{time.time()-t0:.0f}s')

print(f'\nGGraph model saved → {MODEL_GGRAPH}')

## Bước 6 — GGraph: Đánh giá (90 instances)

Eval toàn bộ 9 nhóm × 10 instances = 90 instances.  
Kết quả lưu tại `Results/MyExperimentsGeneral/{folder}/{name}.json`.

**Thời gian ước tính:** ~3–5 giờ (90 instances)

In [ ]:
ggraph_results = {}
print(f"{'#':<4} {'Instance':<26} {'Obj':>8} {'Inf':>8} {'Time':>7}")
print('-' * 58)
idx = 0

for gs in GSIZES:
    folder = gs['folder']
    suffix = gs['suffix']
    inst_base = f"{GGRAPH_BASE}/{folder}"
    out_dir   = f"{OUT_GGRAPH}/{folder}"
    for tp in GTYPES:
        for i in range(10):
            name = f'{tp}{suffix}_G{i}'
            fp   = f'{inst_base}/{name}.graphml'
            if not os.path.exists(fp):
                continue
            idx += 1
            out_path = os.path.join(out_dir, f'{name}.json')
            if os.path.exists(out_path):
                r = load_json(out_path)
                ggraph_results[name] = r
                print(f"{idx:<4} {name:<26} {r['Objective']:>8.3f} "
                      f"{r['Infeasibility']:>8.4f} {'(skip)':>7}")
                continue
            r = eval_instance(fp, name, out_dir, MODEL_GGRAPH)
            ggraph_results[name] = r
            print(f"{idx:<4} {name:<26} {r['Objective']:>8.3f} "
                  f"{r['Infeasibility']:>8.4f} {r['Time Taken']:>6.0f}s")

print(f'\nGGraph eval done: {len(ggraph_results)} instances.')

---
# SO SÁNH BVNS+RL vs VNS

## Bước 7 — Bảng chi tiết và tổng hợp

Ghép kết quả BVNS+RL với VNS baseline theo tên instance.

**Metrics:**
- `Diff_Obj = RL_Obj − VNS_Obj` → âm = RL tốt hơn, dương = VNS tốt hơn
- `Infeasibility = 0` = nghiệm khả thi hoàn toàn

**Đường dẫn VNS:**
```
TGraph : Results/VNSExperiments/{name}_vns.json
GGraph : Results/VNSExperimentsGeneral/{folder}/{name}_vns.json
```

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_rows', 100)

def build_df(rl_dir, vns_dir, names, vns_suffix='_vns'):
    rows = []
    for name in names:
        rl  = load_json(f'{rl_dir}/{name}.json')
        vns = load_json(f'{vns_dir}/{name}{vns_suffix}.json')
        if not (rl and vns):
            continue
        rows.append({
            'Instance':  name,
            'RL_Obj':    round(rl['Objective'],                       3),
            'VNS_Obj':   round(vns['Objective'],                      3),
            'Diff_Obj':  round(rl['Objective'] - vns['Objective'],    3),
            'RL_Inf':    round(rl['Infeasibility'],                   4),
            'VNS_Inf':   round(vns['Infeasibility'],                  4),
            'RL_Time':   round(rl['Time Taken'],                      1),
            'VNS_Time':  round(vns['Time Taken'],                     1),
        })
    return pd.DataFrame(rows)

# TGraph
tg_names = [f'planar{sz}_G{i}' for sz in [500,600,700] for i in range(10)]
df_t = build_df(OUT_PLANAR, VNS_PLANAR, tg_names)

# GGraph (mỗi kích thước)
df_g_list = {}
for gs in GSIZES:
    folder = gs['folder']
    suffix = gs['suffix']
    names  = [f'{tp}{suffix}_G{i}' for tp in GTYPES for i in range(10)]
    df_g_list[folder] = build_df(
        f'{OUT_GGRAPH}/{folder}',
        f'{VNS_GGRAPH}/{folder}',
        names
    )

df_g   = pd.concat(df_g_list.values(), ignore_index=True)
df_all = pd.concat([df_t, df_g], ignore_index=True)

print('=== TGraph (planar 500/600/700) ===')
display(df_t)
print('\n=== GGraph (486/600/726, tất cả loại depot) ===')
display(df_g)

In [ ]:
def print_summary(df, label):
    if df.empty:
        print(f'  [{label}] Chưa có kết quả.')
        return
    n        = len(df)
    rl_wins  = (df['Diff_Obj'] < 0).sum()
    vns_wins = (df['Diff_Obj'] > 0).sum()
    ties     = (df['Diff_Obj'] == 0).sum()
    print(f"\n{'='*62}")
    print(f"  {label}  (n={n})")
    print(f"{'='*62}")
    print(f"  {'Metric':<16} {'BVNS+RL':>10} {'VNS':>10} {'Diff':>10}")
    print(f"  {'-'*50}")
    for col, rc, vc in [('Objective',    'RL_Obj',  'VNS_Obj'),
                         ('Infeasibility','RL_Inf',  'VNS_Inf'),
                         ('Time (s)',     'RL_Time', 'VNS_Time')]:
        rm, vm = df[rc].mean(), df[vc].mean()
        print(f"  {col:<16} {rm:>10.3f} {vm:>10.3f} {rm-vm:>+10.3f}")
    pct = rl_wins / n * 100
    print(f"\n  RL wins: {rl_wins}/{n} ({pct:.0f}%)  VNS wins: {vns_wins}/{n}  Tie: {ties}/{n}")
    print(f"  (Diff_Obj < 0 → RL tốt hơn VNS)")

# ── TGraph ────────────────────────────────────────────────────────
for size in [500, 600, 700]:
    tag = 'in-distribution' if size == 500 else f'generalization +{(size-500)//5}%'
    print_summary(
        df_t[df_t['Instance'].str.startswith(f'planar{size}')],
        f'TGraph — planar{size}  [{tag}]'
    )

# ── GGraph theo kích thước ────────────────────────────────────────
for gs in GSIZES:
    folder = gs['folder']
    suffix = gs['suffix']
    df_sub = df_g_list[folder]
    tag = 'in-distribution (train size)' if suffix == '486' else f'generalization'
    for tp in GTYPES:
        cross = 'in-dist' if tp == 'Center' and suffix == '486' else 'generalization'
        print_summary(
            df_sub[df_sub['Instance'].str.startswith(tp)],
            f'GGraph — {tp}{suffix}  [{folder}, {cross}]'
        )

# ── Tổng ─────────────────────────────────────────────────────────
print_summary(df_all, 'TẤT CA 120 INSTANCES')

## Bước 8 — Biểu đồ

### 8a. Box plot theo nhóm

In [ ]:
# TGraph boxplot
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, size in zip(axes, [500, 600, 700]):
    df_sub = df_t[df_t['Instance'].str.startswith(f'planar{size}')]
    if df_sub.empty:
        ax.set_title(f'planar{size}\n(no data)')
        continue
    bp = ax.boxplot([df_sub['RL_Obj'], df_sub['VNS_Obj']],
                    labels=['BVNS+RL', 'VNS'], patch_artist=True,
                    medianprops=dict(color='red', linewidth=2))
    bp['boxes'][0].set_facecolor('#4a90d9')
    bp['boxes'][1].set_facecolor('#e8a838')
    ax.set_title(f'planar{size}', fontsize=10)
    ax.set_ylabel('Objective')
plt.suptitle('TGraph: BVNS+RL vs VNS — Objective', fontsize=11)
plt.tight_layout()
plt.show()

# GGraph boxplot (theo kích thước)
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, gs in zip(axes, GSIZES):
    df_sub = df_g_list[gs['folder']]
    if df_sub.empty:
        ax.set_title(f"{gs['folder']}\n(no data)")
        continue
    bp = ax.boxplot([df_sub['RL_Obj'], df_sub['VNS_Obj']],
                    labels=['BVNS+RL', 'VNS'], patch_artist=True,
                    medianprops=dict(color='red', linewidth=2))
    bp['boxes'][0].set_facecolor('#4a90d9')
    bp['boxes'][1].set_facecolor('#e8a838')
    ax.set_title(f"{gs['folder']}\n({gs['suffix']} nodes)", fontsize=10)
    ax.set_ylabel('Objective')
plt.suptitle('GGraph: BVNS+RL vs VNS — Objective theo kích thước lưới', fontsize=11)
plt.tight_layout()
plt.show()

### 8b. Scatter: RL vs VNS (dưới y=x → RL tốt hơn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, df_sub, title in [
    (axes[0], df_t,  'TGraph (planar 500/600/700)'),
    (axes[1], df_g,  'GGraph (486/600/726)'),
]:
    if df_sub.empty:
        ax.set_title(f'{title}\n(no data)')
        continue
    mn = min(df_sub[['RL_Obj','VNS_Obj']].min())
    mx = max(df_sub[['RL_Obj','VNS_Obj']].max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1, label='y=x (equal)')
    ax.fill_between([mn,mx],[mn,mn],[mn,mx], alpha=0.07, color='green')
    ax.fill_between([mn,mx],[mn,mx],[mx,mx], alpha=0.07, color='red')
    # màu theo loại instance
    colors = {'500':'#1f77b4','600':'#ff7f0e','700':'#2ca02c',
              '486':'#1f77b4','726':'#2ca02c'}
    for _, row in df_sub.iterrows():
        c = next((v for k,v in colors.items() if k in row['Instance']), 'steelblue')
        ax.scatter(row['VNS_Obj'], row['RL_Obj'], c=c, s=40, alpha=0.8, zorder=3)
    ax.text(mn+(mx-mn)*0.05, mx-(mx-mn)*0.12, 'VNS better', color='red',   fontsize=8)
    ax.text(mx-(mx-mn)*0.35, mn+(mx-mn)*0.03, 'RL better',  color='green', fontsize=8)
    ax.set_xlabel('VNS Objective')
    ax.set_ylabel('BVNS+RL Objective')
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.suptitle('Scatter: BVNS+RL vs VNS Objective', fontsize=12)
plt.tight_layout()
plt.show()

### 8c. Visualize phân vùng

Mỗi màu = 1 district. Dấu **×** = tâm của district đó.

In [ ]:
def plot_solution(graph_path, result_path, ax, title=''):
    G    = nx.read_graphml(graph_path)
    G    = nx.relabel_nodes(G, lambda x: int(x))
    pos  = get_pos(G)  # dùng get_pos từ cell-6: x/y (TGraph) hoặc grid (GGraph)
    data = load_json(result_path)
    if data is None:
        ax.set_title(f'{title}\n(run eval first)')
        return
    districts = {int(k): [int(n) for n in v]
                 for k, v in data['Allocation'].items()}
    palette = plt.cm.tab20  # plt.cm.tab20 tương thích mọi phiên bản matplotlib
    nx.draw_networkx_edges(G, pos, ax=ax, width=0.2, alpha=0.3)
    for k, nodes in districts.items():
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=nodes,
                               node_color=[palette(k % 20)], node_size=15)
        cx = np.mean([pos[n][0] for n in nodes])
        cy = np.mean([pos[n][1] for n in nodes])
        ax.scatter(cx, cy, color='black', marker='x', s=50, zorder=5)
    ax.set_title(f"{title}\nObj={data['Objective']:.3f}  Inf={data['Infeasibility']:.4f}",
                 fontsize=8)
    ax.axis('off')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
samples = [
    (f'{TGRAPH_INST}/planar500_G0.graphml',
     f'{OUT_PLANAR}/planar500_G0.json',            'TGraph: planar500'),
    (f'{TGRAPH_INST}/planar600_G0.graphml',
     f'{OUT_PLANAR}/planar600_G0.json',            'TGraph: planar600'),
    (f'{TGRAPH_INST}/planar700_G0.graphml',
     f'{OUT_PLANAR}/planar700_G0.json',            'TGraph: planar700'),
    (f'{GGRAPH_BASE}/27x27Graphs/Center486_G0.graphml',
     f'{OUT_GGRAPH}/27x27Graphs/Center486_G0.json',  'GGraph: Center486'),
    (f'{GGRAPH_BASE}/30x30Graphs/Center600_G0.graphml',
     f'{OUT_GGRAPH}/30x30Graphs/Center600_G0.json',  'GGraph: Center600'),
    (f'{GGRAPH_BASE}/33x33Graphs/Diagonal726_G0.graphml',
     f'{OUT_GGRAPH}/33x33Graphs/Diagonal726_G0.json','GGraph: Diagonal726'),
]
for ax, (gp, rp, title) in zip(axes.flat, samples):
    plot_solution(gp, rp, ax, title)

plt.suptitle('Kết quả phân vùng BVNS+RL — TGraph & GGraph', fontsize=12)
plt.tight_layout()
plt.show()